# Demo MNIST

A compact MNIST classification setup using GPflow 2 `MultiClass` likelihood.
The example caps the dataset size so the notebook remains a migration smoke test rather than a full experiment.


In [ ]:
import numpy as np
import tensorflow as tf
import gpflow
from gpflow.likelihoods import MultiClass
from gpflow.optimizers import Scipy

from doubly_stochastic_dgp.dgp import DGP

gpflow.config.set_default_float(np.float64)

(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()
X = train_images[:256].reshape(256, -1).astype(np.float64) / 255.0
Y = train_labels[:256, None].astype(np.float64)
Xs = test_images[:64].reshape(64, -1).astype(np.float64) / 255.0
Ys = test_labels[:64, None].astype(np.float64)
Z = X[np.linspace(0, X.shape[0] - 1, 32, dtype=int)].copy()

model = DGP(
    X,
    Y,
    Z,
    [gpflow.kernels.SquaredExponential()],
    MultiClass(10),
    num_outputs=10,
    num_samples=2,
)

loss_before = model.training_loss().numpy()
Scipy().minimize(model.training_loss, model.trainable_variables, options={"maxiter": 1})
class_probabilities, _ = model.predict_y(Xs, 2)
predicted = np.argmax(np.mean(class_probabilities.numpy(), axis=0), axis=1)
accuracy = np.mean(predicted[:, None] == Ys)

float(loss_before), float(accuracy), class_probabilities.shape
